In [ ]:
import ast

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from wildlife_datasets.datasets import TurtlesOfSMSRC

In [ ]:
def negative_exponential(x, a, b):
    return a * (1 - np.exp(-b * x))

def michaelis_menten(x, a, b):
    return (a * x) / (b + x)

def in_intervals(x, intervals):
    for interval in intervals:
        if interval[0] <= x <= interval[1]:
            return True
    return False

def in_tuples(x, ts):
    for t in ts:
        if x in t and len(t) > 1:
            return True
    return False

In [ ]:
ignore_rescue_centers = True
models = [
    (negative_exponential, "negative exponential"),
    (michaelis_menten, "michaelismenten"),
]

In [ ]:
with open("hidden/matched_encounters.txt") as f:
    replace_identity = ast.literal_eval(f.read())

root = "/data/wildlife_datasets/data/TurtlesOfSMSRC"
dataset = TurtlesOfSMSRC(root, replace_identity=replace_identity)

if ignore_rescue_centers:
    with open("hidden/rescue_centers.txt") as f:
        rescue_centers = ast.literal_eval(f.read())

    mask = dataset.metadata["encounter_id"].apply(lambda x: not in_intervals(x, rescue_centers))
    encounters_rc = dataset.metadata.loc[~mask, "encounter_id"].unique()    
    dataset = dataset.get_subset(mask)

    print("Matched rescue centers encounters")
    print(encounters_rc[[in_tuples(x, replace_identity) for x in encounters_rc]])

In [ ]:
len(dataset), dataset.df.encounter_id.nunique(), dataset.df.identity.nunique()

In [ ]:
encounter_counts = [(x, df["encounter_id"].nunique()) for x, df in dataset.metadata.groupby("identity")]
encounter_counts = sorted(encounter_counts, key=lambda x: x[1], reverse=True)
encounter_counts = np.array(encounter_counts)

plt.plot(encounter_counts[:, 1])
plt.xlabel("Individual turtles")
plt.ylabel("Number of encounters")
plt.grid()

In [ ]:
n_super = 3
for i, (identity, _) in enumerate(encounter_counts[:n_super]):
    mask = dataset.metadata["identity"] == identity
    metadata_encounters = dataset.metadata.loc[mask, "encounter_id"]
    print(f"Super resident {i + 1} observed {metadata_encounters.nunique()} times")
    print(metadata_encounters.unique())

In [ ]:
n_individuals = []
encounters = np.sort(dataset.metadata["encounter_id"].unique())
for encounter in encounters:
    mask = dataset.metadata["encounter_id"] <= encounter
    n_individuals.append(dataset.metadata.loc[mask, "identity"].nunique())

xs = np.arange(1, len(n_individuals) + 1)
x_line = np.linspace(1, 5 * max(xs), 200)
title = "without" if ignore_rescue_centers else "with"

for model, model_name in models:
    fit_results, _ = curve_fit(model, xs, n_individuals, maxfev=5000)
    plt.plot(x_line, model(x_line, *fit_results), label=f"Population: estimate ({model_name})")
plt.plot(xs, n_individuals, label="Population: data")
plt.plot(xs, xs, label="Upper bound (identity mapping)", color="black")

plt.title(f"Predicted individuals {title} rescue centers")
plt.xlabel("Number of encounters")
plt.ylabel("Number of individuals")
plt.grid(alpha=0.7)
plt.legend()
plt.savefig(f"estimate-ignore_rc={ignore_rescue_centers}.png", dpi=300, bbox_inches="tight")